In [ ]:
import json
import re
from collections import defaultdict

def parse_sql_to_compressed_string(sql_file_path, output_json_path="schema.json"):
    """
    Parses SQL schema and compresses table schemas into single-line ultra-compact string signatures.
    """
    with open(sql_file_path, "r", encoding="utf-8", errors="ignore") as f:
        sql_content = f.read()

    table_pattern = re.compile(
        r"CREATE\s+TABLE\s+\[dbo\]\.\[(?P<table_name>\w+)\]\s*\((?P<columns_block>.*?)\)\s*ON",
        re.DOTALL | re.IGNORECASE
    )

    column_pattern = re.compile(
        r"^\s*\[(?P<col_name>\w+)\]\s+(?:AS\s+.*|\[(?P<data_type>\w+)\](?:\((?P<type_arg>[\d,\s]+|max)\))?\s*(?P<identity>IDENTITY\(\d+,\d+\))?\s*(?P<nullability>NOT\s+NULL|NULL)?)",
        re.MULTILINE | re.IGNORECASE
    )

    pk_pattern = re.compile(
        r"PRIMARY\s+KEY.*?\((?P<pk_cols>.*?)\)",
        re.DOTALL | re.IGNORECASE
    )

    schema_json = {}

    for table_match in table_pattern.finditer(sql_content):
        table_name = table_match.group("table_name")
        columns_block = table_match.group("columns_block")

        pk_cols = set()
        pk_match = pk_pattern.search(columns_block)
        if pk_match:
            pk_cols = set(re.findall(r"\[(\w+)\]", pk_match.group("pk_cols")))

        columns = []

        for line in columns_block.splitlines():
            line_str = line.strip()
            
            # Computed Columns
            computed_match = re.search(r"\[(\w+)\]\s+AS\s+\((.*?)\)", line_str, re.IGNORECASE)
            if computed_match:
                columns.append((computed_match.group(1), "COMPUTED", ""))
                continue

            # Standard Columns
            col_match = column_pattern.match(line_str)
            if col_match:
                col_name = col_match.group("col_name")
                data_type = col_match.group("data_type").lower()
                type_arg = col_match.group("type_arg")
                identity = bool(col_match.group("identity"))
                nullability = col_match.group("nullability") or ""

                flags = []
                if col_name in pk_cols:
                    flags.append("PK")
                if identity:
                    flags.append("ID")
                if "NOT NULL" in nullability.upper():
                    flags.append("NN")

                flag_str = f"({','.join(flags)})" if flags else ""
                
                # Normalize types
                if data_type in ["nvarchar", "varchar", "char"]:
                    dtype = f"str{type_arg}" if type_arg else "str"
                elif data_type in ["decimal", "numeric"]:
                    dtype = f"dec({type_arg})" if type_arg else "dec"
                else:
                    dtype = data_type

                columns.append((col_name, dtype, flag_str))

        # Group columns dynamically into the single compressed line format
        schema_json[table_name] = compress_columns(columns)

    with open(output_json_path, "w", encoding="utf-8") as out_file:
        json.dump(schema_json, out_file, indent=2)

    print(f"Successfully generated compressed schema JSON at '{output_json_path}'.")


def compress_columns(columns):
    """
    Helper function to aggregate column definitions into a single line format.
    """
    specials = []
    groups = defaultdict(list)
    str50_cols = []

    # Sequence collapse for machine numbers (Machine1Name, Machine2Name -> Machine1-3Name)
    machine_cols = defaultdict(list)

    for col_name, dtype, flags in columns:
        if flags or dtype == "COMPUTED":
            specials.append(f"{col_name}:{dtype}{flags}")
        elif dtype == "str50":
            str50_cols.append(col_name)
        elif re.match(r"^Machine[1-9]", col_name):
            base_key = re.sub(r"\d+", "", col_name)
            machine_cols[(base_key, dtype)].append(col_name)
        else:
            groups[dtype].append(col_name)

    parts = list(specials)

    # Process grouped non-special columns
    for dtype, cols in groups.items():
        # Wildcard grouping for Date, Qty, Bags
        dates = [c for c in cols if c.endswith("Date") or c.endswith("On")]
        qtys = [c for c in cols if "Qty" in c or "Quantity" in c or "KGIn" in c]
        bags = [c for c in cols if "Bag" in c or "Bags" in c]
        
        rest = [c for c in cols if c not in dates and c not in qtys and c not in bags]

        if dates:
            parts.append(f"*Date/*On:{dtype}")
        if qtys:
            parts.append(f"*Qty/*Quantity:{dtype}")
        if bags:
            parts.append(f"*Bags/*Bag:{dtype}")
        
        if rest:
            parts.append(f"{'/'.join(rest)}:{dtype}")

    # Process machine series
    for (base_key, dtype), cols in machine_cols.items():
        num_start = re.search(r"\d+", cols[0]).group()
        num_end = re.search(r"\d+", cols[-1]).group()
        series_name = f"Machine{num_start}-{num_end}{base_key.replace('Machine', '')}"
        parts.append(f"{series_name}:{dtype}")

    # Add 50-length strings in array format
    if str50_cols:
        parts.append(f"[{', '.join(str50_cols)}]:str50")

    return ", ".join(parts)


# Execute
parse_sql_to_compressed_string(
    r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.sql", 
    r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\mes_db_info\schema.json"
)

Successfully generated compressed schema JSON at 'C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json'.


In [ ]:
import json
import re
from collections import defaultdict

def parse_sql_to_compressed_string(sql_file_path, output_json_path="schema.json"):
    """
    Parses PostgreSQL SQL schema and compresses table schemas into single-line ultra-compact string signatures.
    """
    with open(sql_file_path, "r", encoding="utf-8", errors="ignore") as f:
        sql_content = f.read()

    # Updated pattern for PostgreSQL CREATE TABLE public.table_name (...)
    table_pattern = re.compile(
        r"CREATE\s+TABLE\s+(?:public\.)?\"?(?P<table_name>\w+)\"?\s*\((?P<columns_block>.*?)\);",
        re.DOTALL | re.IGNORECASE
    )

    # Pattern for primary key constraint inside columns_block or ALTER TABLE
    pk_pattern = re.compile(
        r"PRIMARY\s+KEY\s*\((?P<pk_cols>.*?)\)",
        re.DOTALL | re.IGNORECASE
    )

    schema_json = {}

    for table_match in table_pattern.finditer(sql_content):
        table_name = table_match.group("table_name")
        columns_block = table_match.group("columns_block")

        # Find primary key columns in table definition or fallback
        pk_cols = set()
        pk_match = pk_pattern.search(columns_block)
        if pk_match:
            pk_cols = set(re.findall(r"\"?(\w+)\"?", pk_match.group("pk_cols")))

        columns = []

        for line in columns_block.splitlines():
            line_str = line.strip().rstrip(",")
            if not line_str or line_str.startswith("--") or line_str.upper().startswith("CONSTRAINT") or line_str.upper().startswith("PRIMARY KEY"):
                continue

            # Match Postgres Column: "col_name" data_type [modifiers] or col_name data_type [modifiers]
            col_match = re.match(
                r"^\"?(?P<col_name>\w+)\"?\s+(?P<data_type>[\w\s]+?)(?:\((?P<type_arg>[\d,\s]+)\))?(?P<rest>.*)$",
                line_str,
                re.IGNORECASE
            )

            if col_match:
                col_name = col_match.group("col_name")
                raw_type = col_match.group("data_type").strip().lower()
                type_arg = col_match.group("type_arg")
                rest = col_match.group("rest") or ""

                # Identify PK, Identity/Serial, NOT NULL
                identity = "nextval" in rest.lower() or "serial" in raw_type
                is_not_null = "not null" in rest.lower()

                flags = []
                if col_name in pk_cols or col_name == "id":  # Defaulting 'id' as PK if common
                    flags.append("PK")
                if identity:
                    flags.append("ID")
                if is_not_null:
                    flags.append("NN")

                flag_str = f"({','.join(flags)})" if flags else ""

                # Normalize types to short representations
                if "character varying" in raw_type or "varchar" in raw_type or "text" in raw_type:
                    dtype = f"str{type_arg}" if type_arg else "str"
                elif "double precision" in raw_type or "numeric" in raw_type or "decimal" in raw_type:
                    dtype = f"dec({type_arg})" if type_arg else "dec"
                elif "integer" in raw_type or "bigint" in raw_type or "smallint" in raw_type:
                    dtype = "int"
                elif "timestamp" in raw_type or "date" in raw_type:
                    dtype = "datetime"
                elif "boolean" in raw_type:
                    dtype = "bool"
                else:
                    dtype = raw_type.split()[0]

                columns.append((col_name, dtype, flag_str))

        # Compress and save table schema
        if columns:
            schema_json[table_name] = compress_columns(columns)

    with open(output_json_path, "w", encoding="utf-8") as out_file:
        json.dump(schema_json, out_file, indent=2)

    print(f"Successfully generated compressed schema JSON at '{output_json_path}'. Total tables processed: {len(schema_json)}")


def compress_columns(columns):
    """
    Helper function to aggregate column definitions into a single line format.
    """
    specials = []
    groups = defaultdict(list)
    str50_cols = []

    machine_cols = defaultdict(list)

    for col_name, dtype, flags in columns:
        if flags or dtype == "COMPUTED":
            specials.append(f"{col_name}:{dtype}{flags}")
        elif dtype == "str50":
            str50_cols.append(col_name)
        elif re.match(r"^Machine[1-9]", col_name):
            base_key = re.sub(r"\d+", "", col_name)
            machine_cols[(base_key, dtype)].append(col_name)
        else:
            groups[dtype].append(col_name)

    parts = list(specials)

    for dtype, cols in groups.items():
        dates = [c for c in cols if c.endswith("Date") or c.endswith("On") or c.endswith("_at") or c.endswith("time")]
        qtys = [c for c in cols if "Qty" in c or "Quantity" in c or "count" in c]
        bags = [c for c in cols if "Bag" in c or "Bags" in c]

        rest = [c for c in cols if c not in dates and c not in qtys and c not in bags]

        if dates:
            parts.append(f"*Date/*On/*_at:{dtype}")
        if qtys:
            parts.append(f"*Qty/*Quantity/*count:{dtype}")
        if bags:
            parts.append(f"*Bags/*Bag:{dtype}")

        if rest:
            parts.append(f"{'/'.join(rest)}:{dtype}")

    for (base_key, dtype), cols in machine_cols.items():
        num_start = re.search(r"\d+", cols[0]).group()
        num_end = re.search(r"\d+", cols[-1]).group()
        series_name = f"Machine{num_start}-{num_end}{base_key.replace('Machine', '')}"
        parts.append(f"{series_name}:{dtype}")

    if str50_cols:
        parts.append(f"[{', '.join(str50_cols)}]:str50")

    return ", ".join(parts)


# Execute
parse_sql_to_compressed_string(
    r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.sql", 
    r"C:\Users\soroj\OneDrive\Desktop\IIIOT Activate Projects\MANUFACTURING-AGENTIC-AI\docs\video_analytics_db_info\construction_ai_schema.json"
)